# ⚠️ Cost Warning

This notebook provisions and uses **two SageMaker real-time endpoints** that incur charges as long as they are running:
- **Quantized endpoint** (`ml.g5.xlarge`): ~$1.41/hr
- **Full-precision endpoint** (`ml.g5.12xlarge`): ~$7.09/hr

**Total: ~$8.50/hr while both endpoints are active.**

Run the **Cleanup** cell at the bottom of this notebook or run `terraform destroy` when you are finished to stop incurring costs.

# Quantized vs Full-Precision Model Comparison: Qwen3-VL-8B-Instruct

Side-by-side comparison of Unsloth's dynamically quantized Qwen3-VL-8B-Instruct (4-bit GGUF via llama.cpp) against the full-precision BF16 variant (via vLLM on SageMaker LMI). Focused on image understanding tasks to showcase vision-language capabilities.

In [ ]:
import json
import time
import base64
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image as IPImage

from comparison_utils import (
    InferenceResult,
    ComparisonResult,
    PRICING,
    calculate_latency,
    calculate_throughput,
    calculate_cost_per_request,
    compute_average_metrics,
    compute_grouped_averages,
    encode_image,
    build_quantized_payload,
    build_full_precision_payload,
    invoke_endpoint,
    run_comparison,
)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Update these endpoint names to match your Terraform outputs
QUANTIZED_ENDPOINT = "qwen3-vl-8b-quantized"
FULL_PRECISION_ENDPOINT = "qwen3-vl-8b-full-precision"
AWS_REGION = "us-east-2"

# Generation parameters
GENERATION_PARAMS = {
    "max_tokens": 512,
    "temperature": 0.1,
}

# Config dict for run_comparison()
CONFIG = {
    "quantized_endpoint": QUANTIZED_ENDPOINT,
    "full_precision_endpoint": FULL_PRECISION_ENDPOINT,
    "aws_region": AWS_REGION,
}

# Metrics store — accumulates results across all prompts
metrics_store: list[ComparisonResult] = []

print(f"Quantized endpoint:      {QUANTIZED_ENDPOINT}")
print(f"Full-precision endpoint: {FULL_PRECISION_ENDPOINT}")
print(f"Region:                  {AWS_REGION}")

: 

## Deployment Configuration Summary

In [ ]:
config_data = {
    "Property": [
        "HuggingFace Model ID",
        "Quantization",
        "Inference Runtime",
        "SageMaker Instance Type",
        "GPU",
        "Model Size on Disk",
        "Model Family",
    ],
    "Quantized Model": [
        "unsloth/Qwen3-VL-8B-Instruct-GGUF",
        "Unsloth Dynamic 2.0 \u2014 4-bit GGUF (Q4_K_M)",
        "llama.cpp (BYOC)",
        "ml.g5.xlarge",
        "NVIDIA A10G (24 GB)",
        "~6.4 GB (LLM + vision projector)",
        "Qwen3-VL (Vision-Language)",
    ],
    "Full-Precision Model": [
        "Qwen/Qwen3-VL-8B-Instruct",
        "None (BF16 full precision)",
        "vLLM (SageMaker LMI)",
        "ml.g5.12xlarge",
        "NVIDIA A10G x4 (96 GB)",
        "~16.4 GB",
        "Qwen3-VL (Vision-Language)",
    ],
}

df_config = pd.DataFrame(config_data)
display(HTML(
    "<p><em>Both models are Qwen3-VL (Vision-Language) variants that natively support "
    "image understanding. No different model family is required for image tasks.</em></p>"
    + df_config.to_html(index=False)
))

## Image-Based Comparisons

Each cell below sends the same image and prompt to both endpoints, displays the input image, shows the model outputs side by side, and plots per-prompt latency and throughput.

In [ ]:
def display_comparison(result: ComparisonResult):
    """Display side-by-side comparison of model outputs with metrics charts."""
    # Display the prompt
    display(HTML(f"<h3>Prompt: {result.prompt_text}</h3>"))
    
    # Display the input image if this is an image prompt
    if result.image_source:
        display(HTML("<h4>Input Image:</h4>"))
        if result.image_source.startswith("http"):
            display(IPImage(url=result.image_source, width=400))
        else:
            display(IPImage(filename=result.image_source, width=400))
    
    # Side-by-side outputs
    q = result.quantized
    fp = result.full_precision
    
    html = f"""
    <table style="width:100%; border-collapse:collapse; margin:10px 0;">
    <tr style="background:#f0f0f0;">
        <th style="padding:10px; border:1px solid #ddd; width:50%;">{q.model_label}</th>
        <th style="padding:10px; border:1px solid #ddd; width:50%;">{fp.model_label}</th>
    </tr>
    <tr>
        <td style="padding:10px; border:1px solid #ddd; vertical-align:top;">
            {q.error if q.error else q.generated_text}
        </td>
        <td style="padding:10px; border:1px solid #ddd; vertical-align:top;">
            {fp.error if fp.error else fp.generated_text}
        </td>
    </tr>
    </table>
    """
    display(HTML(html))
    
    # Skip charts if both errored
    if q.error and fp.error:
        return
    
    # Per-prompt latency and throughput bar charts
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    labels = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
    colors = ["#2196F3", "#FF9800"]
    
    # Latency chart
    latencies = [q.latency_ms, fp.latency_ms]
    axes[0].bar(labels, latencies, color=colors)
    axes[0].set_ylabel("Latency (ms)")
    axes[0].set_title("End-to-End Latency")
    for i, v in enumerate(latencies):
        axes[0].text(i, v + max(latencies) * 0.02, f"{v:.0f}ms", ha="center", fontsize=10)
    
    # Throughput chart
    throughputs = [q.throughput_tps, fp.throughput_tps]
    axes[1].bar(labels, throughputs, color=colors)
    axes[1].set_ylabel("Tokens/sec")
    axes[1].set_title("Token Generation Throughput")
    for i, v in enumerate(throughputs):
        axes[1].text(i, v + max(throughputs) * 0.02, f"{v:.1f}", ha="center", fontsize=10)
    
    plt.tight_layout()
    plt.show()

### 1. Image Description

In [ ]:
# Image Description — Describe the contents of an image in detail
IMAGE_1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
PROMPT_1 = "Describe the contents of this image in detail."

result_1 = run_comparison(PROMPT_1, IMAGE_1, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_1)
display_comparison(result_1)

### 2. Object Identification

In [ ]:
# Object Identification — Identify and list objects in an image
IMAGE_2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/A_small_cup_of_coffee.JPG/640px-A_small_cup_of_coffee.JPG"
PROMPT_2 = "List all distinct objects you can identify in this image."

result_2 = run_comparison(PROMPT_2, IMAGE_2, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_2)
display_comparison(result_2)

### 3. OCR (Text Extraction)

In [ ]:
# OCR — Extract text from an image containing text
IMAGE_3 = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/English_Wikipedia_main_page_screenshot.png/640px-English_Wikipedia_main_page_screenshot.png"
PROMPT_3 = "Extract and transcribe all text visible in this image."

result_3 = run_comparison(PROMPT_3, IMAGE_3, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_3)
display_comparison(result_3)

### 4. Visual Question Answering

In [ ]:
# Visual QA — Answer a specific question about an image
IMAGE_4 = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
PROMPT_4 = "What activity is taking place in this image and where is it happening?"

result_4 = run_comparison(PROMPT_4, IMAGE_4, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_4)
display_comparison(result_4)

### 5. Scene Understanding

In [ ]:
# Scene Understanding — Describe spatial relationships and context
IMAGE_5 = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/10/Empire_State_Building_%28aerial_view%29.jpg/800px-Empire_State_Building_%28aerial_view%29.jpg"
PROMPT_5 = "Describe the spatial layout and relationships between objects in this image."

result_5 = run_comparison(PROMPT_5, IMAGE_5, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_5)
display_comparison(result_5)

## Text-Only Comparison

A single text-only prompt to compare non-visual performance as a secondary baseline.

### 6. Text Generation (No Image)

In [ ]:
# Text-only comparison — no image input
PROMPT_TEXT = "Explain the key benefits of model quantization for production ML deployments."

result_text = run_comparison(PROMPT_TEXT, None, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_text)
display_comparison(result_text)

## Metrics Summary

Aggregated performance metrics across all prompts.

In [ ]:
# Average Latency — Overall and by prompt type
q_avg_lat, fp_avg_lat = compute_average_metrics(metrics_store, "latency_ms")
grouped = compute_grouped_averages(metrics_store)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#2196F3", "#FF9800"]

# Overall average latency
labels_overall = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
vals_overall = [q_avg_lat, fp_avg_lat]
axes[0].bar(labels_overall, vals_overall, color=colors)
axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("Average Latency — All Prompts")
for i, v in enumerate(vals_overall):
    axes[0].text(i, v + max(vals_overall) * 0.02, f"{v:.0f}ms", ha="center", fontsize=10)

# Latency by prompt type (grouped bar chart)
prompt_types = sorted(grouped.keys())
x = range(len(prompt_types))
width = 0.35
q_lats = [grouped[pt]["quantized_avg_latency_ms"] for pt in prompt_types]
fp_lats = [grouped[pt]["full_precision_avg_latency_ms"] for pt in prompt_types]

bars1 = axes[1].bar([i - width/2 for i in x], q_lats, width, label="Quantized", color=colors[0])
bars2 = axes[1].bar([i + width/2 for i in x], fp_lats, width, label="Full Precision", color=colors[1])
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Average Latency — By Prompt Type")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([pt.capitalize() for pt in prompt_types])
axes[1].legend()

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{bar.get_height():.0f}", ha="center", fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{bar.get_height():.0f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Average Throughput
q_avg_thr, fp_avg_thr = compute_average_metrics(metrics_store, "throughput_tps")

fig, ax = plt.subplots(figsize=(7, 5))
labels = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
vals = [q_avg_thr, fp_avg_thr]
colors = ["#2196F3", "#FF9800"]

ax.bar(labels, vals, color=colors)
ax.set_ylabel("Tokens/sec")
ax.set_title("Average Token Generation Throughput — All Prompts")
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.02, f"{v:.1f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

### Cost Comparison

In [ ]:
# Cost Comparison Summary Table
q_hourly = PRICING["ml.g5.xlarge"]
fp_hourly = PRICING["ml.g5.12xlarge"]

q_cost_per_req = calculate_cost_per_request(q_avg_lat, q_hourly)
fp_cost_per_req = calculate_cost_per_request(fp_avg_lat, fp_hourly)

summary_data = {
    "Metric": [
        "Model",
        "Instance Type",
        "Hourly Cost (USD)",
        "Avg Latency (ms)",
        "Avg Throughput (tok/s)",
        "Est. Cost per Request (USD)",
        "Est. Cost per 1K Requests (USD)",
    ],
    "Quantized": [
        "Qwen3-VL-8B \u2014 4-bit GGUF (Q4_K_M)",
        "ml.g5.xlarge",
        f"${q_hourly:.2f}",
        f"{q_avg_lat:.0f}",
        f"{q_avg_thr:.1f}",
        f"${q_cost_per_req:.6f}",
        f"${q_cost_per_req * 1000:.4f}",
    ],
    "Full Precision": [
        "Qwen3-VL-8B \u2014 BF16",
        "ml.g5.12xlarge",
        f"${fp_hourly:.2f}",
        f"{fp_avg_lat:.0f}",
        f"{fp_avg_thr:.1f}",
        f"${fp_cost_per_req:.6f}",
        f"${fp_cost_per_req * 1000:.4f}",
    ],
}

df_summary = pd.DataFrame(summary_data)
display(HTML(df_summary.to_html(index=False)))

## Cleanup

Delete both SageMaker endpoints and their associated resources to stop incurring costs. If any deletion fails, run `terraform destroy` as a fallback.

In [ ]:
# Cleanup — Delete SageMaker endpoints, endpoint configurations, and models
sm_client = boto3.client("sagemaker", region_name=AWS_REGION)

resources_to_delete = [
    ("endpoint", QUANTIZED_ENDPOINT),
    ("endpoint", FULL_PRECISION_ENDPOINT),
    ("endpoint_config", "qwen3-vl-8b-quantized-config"),
    ("endpoint_config", "qwen3-vl-8b-full-precision-config"),
    ("model", "qwen3-vl-8b-quantized"),
    ("model", "qwen3-vl-8b-full-precision"),
]

for resource_type, resource_name in resources_to_delete:
    try:
        if resource_type == "endpoint":
            sm_client.delete_endpoint(EndpointName=resource_name)
        elif resource_type == "endpoint_config":
            sm_client.delete_endpoint_config(EndpointConfigName=resource_name)
        elif resource_type == "model":
            sm_client.delete_model(ModelName=resource_name)
        print(f"✅ Deleted {resource_type}: {resource_name}")
    except sm_client.exceptions.ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "ValidationException" and "Could not find" in str(e):
            print(f"ℹ️  {resource_type} '{resource_name}' already deleted.")
        else:
            print(f"❌ Failed to delete {resource_type} '{resource_name}': {e}")
            print("   Fallback: run 'terraform destroy' from the terraform/ directory.")
    except Exception as e:
        print(f"❌ Failed to delete {resource_type} '{resource_name}': {e}")
        print("   Fallback: run 'terraform destroy' from the terraform/ directory.")

print("\nCleanup complete. If any deletions failed, run 'terraform destroy' as a fallback.")